# A beginner's guide to your production system
### Learn by investigating a missing museum jewel

This notebook uses **your uploaded `reasoning.engine`**, without replacing its algorithms. Work from top to bottom. Each example explains what to expect and includes checks where useful.

**You will learn:** represent facts and rules; manage working memory; match variables; combine conditions; derive new facts with forward chaining; answer questions with backward chaining; and understand the current limits of the engine.

The mystery is a deliberately simplified logical world. A conclusion follows from the supplied rules and facts; it is not evidence that these rules are valid for real criminal investigations.

## 1. Setup
Keep this notebook in your project, select the Python environment where `reasoning` is installed, and run **Restart Kernel → Run All**. If `reasoning` lives under `src`, install your existing package from the project root with `python -m pip install -e .` (assuming your project already has packaging configured). You do not need to paste the engine into the notebook.

Only the Python standard library and your module are used. Outputs are included so you can also read the tutorial without executing it.

In [1]:
from reasoning import engine
from reasoning.engine import (
    WorkingMemory, is_var, unify, substitute, freshen_rule,
    match_conditions, forward_chain, prove_conditions, backward_chain,
    backward_answers, applicable, step,
)
from contextlib import redirect_stdout
from io import StringIO

print("Loaded:", engine.__name__)

Loaded: reasoning.engine


## 2. The three parts of a production system

| Part | Meaning | In this engine |
|---|---|---|
| Working memory | Facts currently recorded | `WorkingMemory` |
| Rule base | IF conditions THEN conclusion | A list of dictionaries |
| Inference engine | Matches rules and finds conclusions | `forward_chain`, `backward_chain` |

A **fact** is a flat tuple such as `("at", "alice", "gallery")`. The first item names the relationship (predicate); the remaining items are its arguments. This says Alice is at the gallery. A **ground** fact contains no variables.

A **pattern** may contain variables: `("at", "?person", "gallery")` asks who is at the gallery. Variables are strings starting with `?`. The engine operates on tuples; it does not understand ordinary English or infer meanings from names.

In [2]:
for term in ["?person", "alice", "at", 7]:
    print(repr(term), "is a variable:", is_var(term))
assert is_var("?person") and not is_var("alice")

'?person' is a variable: True
'alice' is a variable: False
'at' is a variable: False
7 is a variable: False


## 3. Store, inspect, and change facts
`WorkingMemory(facts)` constructs memory; `repr(wm)` invokes `__repr__` to show its size. The public `facts` attribute is a set, so duplicates disappear and iteration order is unspecified. Use `sorted` for readable output when your facts have mutually comparable terms, as these string-only examples do.

`tell(fact)` adds a fact. `holds(fact)` checks exact membership; it does not perform reasoning and does not interpret variables as wildcards. `retract(fact)` removes a fact and safely ignores an absent fact. These change methods return `None`.

In [3]:
wm = WorkingMemory([("at", "alice", "gallery")])
print(wm)
wm.tell(("at", "bob", "garden"))
wm.tell(("at", "bob", "garden"))
print("Facts:", sorted(wm.facts))
print("Alice at gallery?", wm.holds(("at", "alice", "gallery")))
print("Pattern checked with holds:", wm.holds(("at", "?person", "gallery")))
wm.retract(("at", "bob", "garden"))
wm.retract(("at", "nobody", "roof"))
assert len(wm.facts) == 1

WorkingMemory(1 facts)
Facts: [('at', 'alice', 'gallery'), ('at', 'bob', 'garden')]
Alice at gallery? True
Pattern checked with holds: False


### Save and restore a state
`snapshot()` returns a copy of the fact set. `restore(saved)` replaces the current facts with a copy of that saved set. This supports trying an action and undoing it. It does not save rules, Python variables, or the fresh-variable counter.

In [4]:
saved = wm.snapshot()
wm.tell(("holding", "alice", "jewel"))
print("During experiment:", sorted(wm.facts))
wm.restore(saved)
print("After restore:", sorted(wm.facts))
assert not wm.holds(("holding", "alice", "jewel"))
assert wm.facts == saved

During experiment: [('at', 'alice', 'gallery'), ('holding', 'alice', 'jewel')]
After restore: [('at', 'alice', 'gallery')]


## 4. Unification: what must the variables equal?
`unify(pattern, fact, subst)` tries to make two tuples equal while respecting existing bindings. Either side may contain variables. It returns a **new binding dictionary** on success and `None` on failure. The supplied dictionary is not changed.

`{}` is a successful match requiring no bindings. It is different from `None`. Therefore use `result is not None` to check success, rather than `if result`.

In [5]:
pattern = ("at", "?person", "?room")
binding = unify(pattern, ("at", "alice", "gallery"), {})
print("Successful match:", binding)
print("Constant conflict:", unify(("at", "alice", "roof"),
                                  ("at", "alice", "gallery"), {}))
print("Ground success:", unify(("ready", "alice"), ("ready", "alice"), {}))
prior = {"?person": "bob"}
print("Existing binding conflict:", unify(pattern, ("at", "alice", "gallery"), prior))
print("Original dictionary:", prior)
assert prior == {"?person": "bob"}
assert unify(("p",), ("p",), {}) == {}
assert unify(("p",), ("p", "a"), {}) is None

Successful match: {'?person': 'alice', '?room': 'gallery'}
Constant conflict: None
Ground success: {}
Existing binding conflict: None
Original dictionary: {'?person': 'bob'}


### Repeated variables impose equality
In `("near", "?x", "?x")`, both positions must refer to the same object. Different variables are **allowed** to have the same value; they do not automatically mean different objects. There is no built-in inequality operator.

In [6]:
print("Same object:", unify(("near", "?x", "?x"), ("near", "alice", "alice"), {}))
print("Different objects:", unify(("near", "?x", "?x"), ("near", "alice", "bob"), {}))
print("Variable on each side:", unify(("holding", "?a"), ("holding", "?b"), {}))

Same object: {'?x': 'alice'}
Different objects: None
Variable on each side: {'?a': '?b'}


## 5. Substitution: use the answer in another pattern
`substitute(pattern, subst)` fills in bound variables. It leaves unbound variables untouched. This is how a general conclusion becomes a specific fact.

`_walk(term, subst)` is a private helper that follows binding chains. If `?x → ?y` and `?y → alice`, the final value of `?x` is `alice`. Normally call `substitute`, rather than `_walk`, in application code. Avoid manually creating cyclic bindings such as `?x → ?y → ?x`; `_walk` has no cycle guard.

In [7]:
print("New fact:", substitute(("seen", "?person", "?room"), binding))
print("Partial:", substitute(("at", "?person", "?room"), {"?person": "alice"}))
chain = {"?x": "?y", "?y": "alice"}
print("Private helper:", engine._walk("?x", chain))
print("Through substitute:", substitute(("suspect", "?x"), chain))
assert substitute(("suspect", "?x"), chain) == ("suspect", "alice")

New fact: ('seen', 'alice', 'gallery')
Partial: ('at', 'alice', '?room')
Private helper: alice
Through substitute: ('suspect', 'alice')


## 6. Query stored facts with `ask`
`wm.ask(pattern, subst=None)` yields one binding dictionary for each matching stored fact. It is a **generator**: use `list(...)` to collect the answers, or loop over it. A generator produces values as requested and is exhausted after consumption.

For a ground question, `[{}]` means success and `[]` means failure. This function only searches recorded facts. It does not consult rules.

In [8]:
wm = WorkingMemory([
    ("at", "alice", "gallery"),
    ("at", "bob", "gallery"),
    ("at", "cara", "garden"),
])
print("People at gallery:", sorted(b["?person"] for b in wm.ask(("at", "?person", "gallery"))))
print("Ground success:", list(wm.ask(("at", "alice", "gallery"))))
print("Ground failure:", list(wm.ask(("at", "alice", "roof"))))
print("Existing binding:", list(wm.ask(("at", "?person", "?room"), {"?person": "cara"})))
answers = wm.ask(("at", "?person", "gallery"))
print("First consumption:", len(list(answers)))
print("Second consumption:", list(answers))

People at gallery: ['alice', 'bob']
Ground success: [{}]
Ground failure: []
Existing binding: [{'?person': 'cara', '?room': 'garden'}]
First consumption: 2
Second consumption: []


## 7. Rules and AND conditions
A rule has an `"if"` list of conditions and a `"then"` tuple. All conditions must hold with **consistent bindings**. Multiple rules offering the same conclusion provide alternative ways to establish it (OR).

This rule says: if a person was in the gallery and has access to the cabinet, that person had an opportunity. Use the same variable name across conditions to refer to the same person. Predicates such as `has_access` are ordinary names with no special implementation.

In [9]:
opportunity_rule = {
    "if": [("at", "?person", "gallery"), ("has_access", "?person", "cabinet")],
    "then": ("opportunity", "?person"),
}
wm.tell(("has_access", "alice", "cabinet"))
wm.tell(("has_access", "cara", "cabinet"))
print(opportunity_rule)

{'if': [('at', '?person', 'gallery'), ('has_access', '?person', 'cabinet')], 'then': ('opportunity', '?person')}


### `match_conditions`: join matches already in memory
`match_conditions(wm, conditions, subst=None)` recursively matches the first condition, carries each binding into the remaining conditions, and yields only complete matches.

Here the location condition allows Alice or Bob. The access condition allows Alice or Cara. Sharing `?person` means only Alice satisfies both. No rule has fired yet.

The recursion ends when there are no conditions left. An empty condition list succeeds once with the current binding: every one of zero required conditions is satisfied.

In [10]:
print("All complete matches:", list(match_conditions(wm, opportunity_rule["if"])))
print("Bob fixed in advance:", list(match_conditions(wm, opportunity_rule["if"], {"?person": "bob"})))
print("No conditions:", list(match_conditions(wm, [])))
assert list(match_conditions(wm, opportunity_rule["if"])) == [{"?person": "alice"}]
assert not wm.holds(("opportunity", "alice"))

All complete matches: [{'?person': 'alice'}]
Bob fixed in advance: []
No conditions: [{}]


## 8. Build the museum mystery
We will use the same facts and rules for both reasoning directions. In this toy world, opportunity plus motive produces a suspect; a suspect whose fingerprint is on the cabinet becomes a person to investigate. We deliberately avoid concluding guilt.

Alice has opportunity and motive. Bob has motive but lacks cabinet access. Cara has access but was elsewhere. Only Alice has the recorded fingerprint.

Every variable in a conclusion must be bound by the conditions for these examples to produce ground facts. The engine does not validate this requirement for you.

In [11]:
DEMO_FACTS = [
    ("at", "alice", "gallery"), ("at", "bob", "gallery"),
    ("at", "cara", "garden"),
    ("has_access", "alice", "cabinet"), ("has_access", "cara", "cabinet"),
    ("motive", "alice"), ("motive", "bob"),
    ("fingerprint_on", "alice", "cabinet"),
]
DEMO_RULES = [
    opportunity_rule,
    {"if": [("opportunity", "?person"), ("motive", "?person")],
     "then": ("suspect", "?person")},
    {"if": [("suspect", "?person"), ("fingerprint_on", "?person", "cabinet")],
     "then": ("investigate", "?person")},
]
print("Initial facts:", len(DEMO_FACTS), "Rules:", len(DEMO_RULES))
for number, rule in enumerate(DEMO_RULES, 1):
    print(number, rule)

Initial facts: 8 Rules: 3
1 {'if': [('at', '?person', 'gallery'), ('has_access', '?person', 'cabinet')], 'then': ('opportunity', '?person')}
2 {'if': [('opportunity', '?person'), ('motive', '?person')], 'then': ('suspect', '?person')}
3 {'if': [('suspect', '?person'), ('fingerprint_on', '?person', 'cabinet')], 'then': ('investigate', '?person')}


## 9. Forward chaining: start with facts and derive consequences
The question is **“What else follows from everything we currently know?”**

Your engine repeats these steps:
1. Match each rule's conditions against the current memory.
2. Substitute every complete binding into the conclusion.
3. Collect conclusions that are not already recorded.
4. Add all collected conclusions after the complete pass through the rules.
5. Stop when a pass produces nothing new (a **fixed point**).

Because your implementation batches additions at the end of a pass, a newly derived fact is available to other rules starting in the **next** pass.

| Pass | New facts |
|---|---|
| 1 | `("opportunity", "alice")` |
| 2 | `("suspect", "alice")` |
| 3 | `("investigate", "alice")` |
| 4 | Nothing; stop |

The following teaching loop displays these passes using the engine's matching functions. It mirrors the batching in your `forward_chain`; it does not replace that function.

In [12]:
trace_wm = WorkingMemory(DEMO_FACTS)
pass_number = 0
while True:
    pass_number += 1
    additions = set()
    for rule in DEMO_RULES:
        for b in match_conditions(trace_wm, rule["if"]):
            conclusion = substitute(rule["then"], b)
            if not trace_wm.holds(conclusion):
                additions.add(conclusion)
    print(f"Pass {pass_number}: {sorted(additions)}")
    if not additions:
        break
    for fact in additions:
        trace_wm.tell(fact)

Pass 1: [('opportunity', 'alice')]
Pass 2: [('suspect', 'alice')]
Pass 3: [('investigate', 'alice')]
Pass 4: []


### Call the actual `forward_chain`
**Implementation detail:** the docstring says it returns memory, but the uploaded function has no `return` statement. Its result is `None`. Keep using the original `wm` object after calling it; do not write `wm = forward_chain(wm, rules)`.

In [13]:
forward_wm = WorkingMemory(DEMO_FACTS)
result = forward_chain(forward_wm, DEMO_RULES)
print("Return value:", result)
print("Derived facts:", sorted(forward_wm.facts - set(DEMO_FACTS)))
assert result is None
assert forward_wm.facts == trace_wm.facts
assert forward_wm.holds(("investigate", "alice"))
assert not forward_wm.holds(("investigate", "bob"))
before = forward_wm.snapshot()
forward_chain(forward_wm, DEMO_RULES)
assert forward_wm.facts == before
print("Running again adds no facts.")

Return value: None
Derived facts: [('investigate', 'alice'), ('opportunity', 'alice'), ('suspect', 'alice')]
Running again adds no facts.


## 10. Backward chaining: start with a question
The question is **“Can I establish this particular goal?”**

For `("investigate", "alice")`, the engine can work backwards through these dependencies:

| Goal | What would establish it? |
|---|---|
| Investigate Alice | Suspect Alice AND Alice's fingerprint |
| Suspect Alice | Opportunity Alice AND motive Alice |
| Opportunity Alice | Alice at gallery AND Alice has cabinet access |
| Remaining conditions | Match the initial facts |

`backward_chain(wm, rules, query, subst=None)` first yields direct matches from memory. It then tries each rule whose conclusion unifies with the query, freshens the rule's variables, and recursively proves all its conditions. Every completed proof yields a binding dictionary. Failure yields no answers.

It **does not add its conclusions to memory**. Several proofs may yield the same answer. The implementation prints goals and matching rules as it searches; these are search activity, not a list of established conclusions.

In [14]:
backward_wm = WorkingMemory(DEMO_FACTS)
before = backward_wm.snapshot()
# One live trace, using your engine's own debug prints.
proofs = list(backward_chain(backward_wm, DEMO_RULES, ("investigate", "alice")))
print("Number of successful proofs:", len(proofs))
print("Proven?", bool(proofs))
print("Conclusion stored?", backward_wm.holds(("investigate", "alice")))
assert proofs
assert backward_wm.facts == before

('investigate', 'alice')
('investigate', 'alice') matches binding: {'?person__2': 'alice'} in rule {'then': ('investigate', '?person__2'), 'if': [('suspect', '?person__2'), ('fingerprint_on', '?person__2', 'cabinet')]}
('suspect', '?person__2')
('suspect', '?person__2') matches binding: {'?person__2': 'alice', '?person__4': 'alice'} in rule {'then': ('suspect', '?person__4'), 'if': [('opportunity', '?person__4'), ('motive', '?person__4')]}
('opportunity', '?person__4')
('opportunity', '?person__4') matches binding: {'?person__2': 'alice', '?person__4': 'alice', '?person__5': 'alice'} in rule {'then': ('opportunity', '?person__5'), 'if': [('at', '?person__5', 'gallery'), ('has_access', '?person__5', 'cabinet')]}
('at', '?person__5', 'gallery')
('has_access', '?person__5', 'cabinet')
('motive', '?person__4')
('fingerprint_on', '?person__2', 'cabinet')
Number of successful proofs: 1
Proven? True
Conclusion stored? False


### Ask “Who?” and read useful answers
Proof dictionaries may include internal fresh variable names and variable-to-variable links. Use `substitute(query, binding)` to reconstruct the answer rather than interpreting every internal entry. A set removes duplicate answers. Converting the generator to a list actually executes the search.

Here we capture debug printing to keep the output focused; the reasoning itself is unchanged.

In [15]:
query = ("investigate", "?who")
with redirect_stdout(StringIO()):
    proofs = list(backward_chain(backward_wm, DEMO_RULES, query))
answers = {substitute(query, b) for b in proofs}
print("Answers:", sorted(answers))
assert answers == {("investigate", "alice")}
with redirect_stdout(StringIO()):
    bob_proofs = list(backward_chain(backward_wm, DEMO_RULES, ("investigate", "bob")))
print("Bob proven?", bool(bob_proofs))

Answers: [('investigate', 'alice')]
Bob proven? False


## 11. Why `prove_conditions` differs from `match_conditions`
Both functions establish a conjunction and pass consistent bindings along the conditions. Their difference is how each condition is established:

| Function | How it handles each condition |
|---|---|
| `match_conditions(wm, conditions, subst=None)` | Searches stored facts via `wm.ask` |
| `prove_conditions(wm, rules, conditions, subst=None)` | Also searches rules via `backward_chain` |

Before forward chaining, `opportunity` is not stored. Consequently matching the suspect rule's conditions fails, but proving those conditions can succeed by deriving opportunity along the way. The empty list base case in both functions yields one successful binding.

In [16]:
conditions = DEMO_RULES[1]["if"]
print("Match recorded conditions:", list(match_conditions(backward_wm, conditions)))
with redirect_stdout(StringIO()):
    proved = list(prove_conditions(backward_wm, DEMO_RULES, conditions))
print("People satisfying proved conditions:", sorted({
    substitute(("person", "?person"), b) for b in proved
}))
print("Empty conjunction:", list(prove_conditions(backward_wm, DEMO_RULES, [])))
assert proved

Match recorded conditions: []
People satisfying proved conditions: [('person', 'alice')]
Empty conjunction: [{}]


## 12. Why `freshen_rule` renames variables
A variable belongs to one use of a rule. If two uses share the same Python string `?person`, bindings from one use could accidentally restrict the other. **Standardizing variables apart** gives each use its own names, while preserving repeated occurrences within that use.

`freshen_rule(rule)` returns a new dictionary with consistent names such as `?person__42`. `_rule_ids` is an internal `itertools.count()` counter; suffix numbers depend on earlier searches. Their precise numbers do not affect the logical answer. Renaming does not modify the original rule and does not itself prove anything.

In [17]:
fresh_a = freshen_rule(opportunity_rule)
fresh_b = freshen_rule(opportunity_rule)
print("Original:", opportunity_rule)
print("First use:", fresh_a)
print("Second use:", fresh_b)
assert fresh_a["then"][1] != fresh_b["then"][1]
assert fresh_a["then"][1] == fresh_a["if"][0][1] == fresh_a["if"][1][1]
assert opportunity_rule["then"] == ("opportunity", "?person")

Original: {'if': [('at', '?person', 'gallery'), ('has_access', '?person', 'cabinet')], 'then': ('opportunity', '?person')}
First use: {'then': ('opportunity', '?person__75'), 'if': [('at', '?person__75', 'gallery'), ('has_access', '?person__75', 'cabinet')]}
Second use: {'then': ('opportunity', '?person__76'), 'if': [('at', '?person__76', 'gallery'), ('has_access', '?person__76', 'cabinet')]}


## 13. Use your `backward_answers` helper
Your helper has the signature `backward_answers(wm, query)` and looks up `DEMO_RULES` in the **engine module's global namespace**. Defining `DEMO_RULES` only in this notebook does not supply that global.

To demonstrate the uploaded helper without editing your file, the next cell temporarily sets `engine.DEMO_RULES`, calls the helper, and restores the previous module state. The helper returns a set of instantiated queries, but the underlying engine still prints debug messages; “display-independent” describes the answer format, not silent execution.

For reusable application code, a possible future change is to accept an explicit rule argument: `backward_answers(wm, rules, query)`. That change has **not** been applied here.

In [18]:
_missing = object()
_previous_rules = getattr(engine, "DEMO_RULES", _missing)
try:
    engine.DEMO_RULES = DEMO_RULES
    with redirect_stdout(StringIO()):
        helper_answers = backward_answers(backward_wm, ("investigate", "?who"))
    print("Helper answers:", sorted(helper_answers))
    assert helper_answers == {("investigate", "alice")}
finally:
    if _previous_rules is _missing:
        del engine.DEMO_RULES
    else:
        engine.DEMO_RULES = _previous_rules

Helper answers: [('investigate', 'alice')]


## 14. Two directions, the same rule meaning

| Feature | Forward chaining | Backward chaining |
|---|---|---|
| Starting point | All currently stored facts | A particular query |
| Work performed | Derives all reachable conclusions | Searches proof paths relevant to the goal |
| Memory | Adds conclusions | Leaves memory unchanged |
| Output in this upload | `None`; inspect changed memory | Generator of proof bindings |
| Good use | Reuse many consequences across queries | Answer a focused question |

With these finite, positive, acyclic rules, querying the forward closure and asking backward from the initial facts gives the same answer set. Forward chaining may do extra work for unrelated predicates. Backward chaining may repeat work across questions; this implementation has no answer cache.

In [19]:
query = ("investigate", "?who")
forward_answers = {substitute(query, b) for b in forward_wm.ask(query)}
with redirect_stdout(StringIO()):
    backward_answer_set = {substitute(query, b)
                           for b in backward_chain(backward_wm, DEMO_RULES, query)}
print("Forward:", sorted(forward_answers))
print("Backward:", sorted(backward_answer_set))
assert forward_answers == backward_answer_set

Forward: [('investigate', 'alice')]
Backward: [('investigate', 'alice')]


## 15. Important limits: missing facts, retraction, and loops

**Missing is not explicit negation.** If the engine cannot prove Bob should be investigated, that means the supplied facts and rules do not establish the query. It does not establish a logical `not investigate Bob`. There is no built-in negation, arithmetic, inequality, or confidence model. Everything is flat tuple matching.

**Forward reasoning here is monotonic:** it adds facts. `retract` can remove a fact manually, but the engine does not automatically remove consequences that depended on it. For a changed evidence set, rebuild memory from the revised base facts and derive consequences again. Keeping asserted evidence separate from derived facts makes this possible.

In [20]:
stale_wm = WorkingMemory(DEMO_FACTS)
forward_chain(stale_wm, DEMO_RULES)
stale_wm.retract(("motive", "alice"))
print("Old derived suspect remains:", stale_wm.holds(("suspect", "alice")))
revised_facts = [f for f in DEMO_FACTS if f != ("motive", "alice")]
rebuilt_wm = WorkingMemory(revised_facts)
forward_chain(rebuilt_wm, DEMO_RULES)
print("After rebuilding, suspect:", rebuilt_wm.holds(("suspect", "alice")))
assert not rebuilt_wm.holds(("investigate", "alice"))

Old derived suspect remains: True
After rebuilding, suspect: False


### Termination and safe examples
Forward chaining stops when no new facts can be added. A finite domain and rules that do not invent new values keep the set of possible ground facts finite in these examples.

**The uploaded backward engine has no recursion limit, visited-goal check, or tabling**, despite its docstring asking for a bound. Freshening prevents name clashes; it does not prevent infinite proof searches. Cyclic rules such as `p(?x) → q(?x)` and `q(?x) → p(?x)` can recurse indefinitely. Even a found answer does not guarantee that fully consuming the generator will finish if another branch loops. This notebook uses acyclic rules and does not run that hazardous example.

Also keep working-memory facts ground and flat. The class does not enforce that invariant. Conclusions with unbound variables are a modelling error for the intended fact representation.

## 16. Time and actions: what is present and what is unfinished?
`applicable(wm, action, t)` and `step(wm, action, t)` are declared but raise `NotImplementedError` in this upload. No action dictionary format has yet been implemented, so the calls below deliberately catch that exception rather than pretending these features work.

The intended distinction is:

- `applicable` would find action instances whose preconditions hold at time `t`.
- `step` would add effects at `t+1`, carry forward facts not terminated, and retain past facts for history queries.

For example, a future “grab jewel” action might terminate “jewel at gallery” and “hand empty”, add “holding jewel”, and preserve “door locked”. Those are intended semantics from the docstrings, not capabilities of the current functions. A positive rule that concludes a new fact does not automatically remove the old location.

In [21]:
for function in (applicable, step):
    try:
        function(WorkingMemory(), {}, 0)
    except NotImplementedError:
        print(function.__name__ + ": placeholder; not implemented in this engine")

applicable: placeholder; not implemented in this engine
step: placeholder; not implemented in this engine


## 17. Practice: change the evidence
**Predict before running:**
1. Add Bob's access to the cabinet. Which conclusions should appear?
2. Then add Bob's fingerprint. What changes?
3. Why does Cara still not become a suspect?

Use a fresh memory for each experiment so earlier derived facts do not contaminate the results.

In [22]:
experiment_1 = WorkingMemory(DEMO_FACTS + [("has_access", "bob", "cabinet")])
forward_chain(experiment_1, DEMO_RULES)
print("Experiment 1 new facts:", sorted(experiment_1.facts - set(DEMO_FACTS)))
experiment_2 = WorkingMemory(DEMO_FACTS + [
    ("has_access", "bob", "cabinet"), ("fingerprint_on", "bob", "cabinet")
])
forward_chain(experiment_2, DEMO_RULES)
print("Experiment 2 investigate:", sorted(
    substitute(("investigate", "?who"), b)
    for b in experiment_2.ask(("investigate", "?who"))
))
assert experiment_1.holds(("suspect", "bob"))
assert not experiment_1.holds(("investigate", "bob"))
assert experiment_2.holds(("investigate", "bob"))

Experiment 1 new facts: [('has_access', 'bob', 'cabinet'), ('investigate', 'alice'), ('opportunity', 'alice'), ('opportunity', 'bob'), ('suspect', 'alice'), ('suspect', 'bob')]
Experiment 2 investigate: [('investigate', 'alice'), ('investigate', 'bob')]


### Answers and an extension
1. Bob gains opportunity and becomes a suspect, but lacks the recorded fingerprint.
2. Bob now also satisfies the investigation rule.
3. Cara lacks both gallery presence and motive in the supplied evidence.

**Extension:** add a rule that concludes `("interview", "?person")` from `("suspect", "?person")`. Try forward chaining and backward chaining. Before running, decide whether an interview should also require the fingerprint. Your rule definition determines the answer.

In [23]:
interview_rule = {"if": [("suspect", "?person")], "then": ("interview", "?person")}
extended_rules = DEMO_RULES + [interview_rule]
exercise_wm = WorkingMemory(DEMO_FACTS)
with redirect_stdout(StringIO()):
    interviews = {substitute(("interview", "?who"), b)
                  for b in backward_chain(exercise_wm, extended_rules, ("interview", "?who"))}
print("Interview answers:", sorted(interviews))
assert interviews == {("interview", "alice")}

Interview answers: [('interview', 'alice')]


## 18. Function reference and troubleshooting

| Function or method | Purpose | Demonstrated in |
|---|---|---|
| `is_var(t)` | Recognize `?` variables | 2 |
| `_walk(t, s)` (private) | Resolve a chain of bindings | 5 |
| `unify(pattern, fact, subst)` | Find consistent bindings or return `None` | 4 |
| `substitute(pattern, subst)` | Instantiate a tuple | 5 |
| `WorkingMemory(facts=())` / `__init__`, `__repr__()` | Construct memory and summarize its size | 3 |
| `tell(fact)`, `retract(fact)` | Add or discard a fact | 3 |
| `holds(fact)` | Exact fact membership | 3 |
| `ask(pattern, subst=None)` | Yield matches from stored facts | 6 |
| `snapshot()`, `restore(snap)` | Save and restore fact sets | 3 |
| `match_conditions(wm, conditions, subst=None)` | Match an AND against stored facts | 7 |
| `forward_chain(wm, rules)` | Add derived facts to a fixed point; returns `None` | 9 |
| `backward_chain(wm, rules, query, subst=None)` | Yield proofs from facts and rules | 10 |
| `prove_conditions(wm, rules, conditions, subst=None)` | Recursively prove an AND | 11 |
| `freshen_rule(rule)` | Give each rule use its own variable names | 12 |
| `backward_answers(wm, query)` | Return unique instantiated answers using module `DEMO_RULES` | 13 |
| `applicable(wm, action, t)`, `step(wm, action, t)` | Currently raise `NotImplementedError` | 16 |

| Symptom | Likely cause or next check |
|---|---|
| `ModuleNotFoundError: reasoning` | Select the project environment; install the existing package if needed |
| `NameError: DEMO_RULES` from the helper | Rules are not defined in the engine module; see section 13 |
| Memory becomes `None` | You assigned the return of `forward_chain`; keep the original object |
| `<generator object ...>` | Consume it with `list(...)`, a loop, or a set comprehension |
| A ground success appears to fail | Check `is not None`; `{}` is a successful binding |
| Unexpected variable suffixes | Freshening is normal; use `substitute` to read the answer |
| Repeated answers | Several proofs can establish the same fact; deduplicate instantiated tuples |
| A withdrawn clue leaves old conclusions | Rebuild from updated base evidence; there is no truth-maintenance system |
| Search never ends | Check for recursive or cyclic proof paths; no recursion guard is implemented |

The useful sequence to remember is **match → obtain bindings → substitute → conclude**. Forward chaining repeats that sequence across the fact base. Backward chaining starts with a desired conclusion and recursively asks which conditions could establish it.